In [1]:
# to import things from script folder on this notebooks
import sys; sys.path.append('..')

from Bio import AlignIO, SeqIO
import pandas as pd
from pathlib import Path

from scripts._Mafft import MafftCommandline
from scripts.conf import DATA_RAW, DATA_INTERIM, DATA_PROCESSED

In [2]:
rawfile = "allprot0922.fasta"
seqsfile = DATA_RAW/rawfile
# seq id for Wuhan reference sample
reference_seq = "EPI_ISL_402124"

In [3]:
prot_names = ["NSP1","NSP2","NSP3","NSP4","NSP5","NSP6","NSP7","NSP8","NSP9","NSP10","NSP11","NSP12",
              "NSP13","NSP14","NSP15","NSP16","Spike","NS3","E","M","NS6","NS7a","NS7b","NS8","N"]
len(prot_names)

25

In [14]:
# Parse rawfasta and separate proteins as reference vs others. This take aprox 1-2 minutes
records = SeqIO.parse(seqsfile,"fasta")


ref_seqs = dict()
other_seqs = dict()
for i,r in enumerate(records,1):
    head = r.description
    head_sections = head.split("|")
    # relevant header sections
    prot_name = head_sections[0]
    sample_id = head_sections[3]
    
    if reference_seq == sample_id:
        ref_seqs[prot_name+"|"+head_sections[1]] = str(r.seq)
    else:
        other_seqs[prot_name+"|"+head_sections[1]] = str(r.seq)

    

### Write reference seq proteins

In [4]:
sars2protein_folder = DATA_INTERIM/'sars2_proteins'
refoutfile = 'sars2_refWuhan_{}.fasta'


In [18]:

if not sars2protein_folder.is_dir():
    sars2protein_folder.mkdir()

for k,v in ref_seqs.items():
    name = k.split("|")[0]
    
    with open(sars2protein_folder/refoutfile.format(name),'w') as f:
        f.write('>{}\n'.format(k))
        f.write('{}\n'.format(v))

### Write all other seqs proteins to respective fasta file

Using multiproccesing with ThreadPoolExecutor to help us fast processing.
You can adjust the number of CPUs used with `max_worker` argument from `ThreadPoolExecutor`.

In [20]:
from concurrent.futures import ProcessPoolExecutor,ThreadPoolExecutor

In [22]:

def write_proteins(d, protein):
    
    """
    Select specific protein from seqs dictionary
    and write to respective fasta.
    
    """
    
    selected = filter(lambda items: items[0].split('|')[0]==protein, d.items())
    
    with open(sars2protein_folder/'sars2_others_{}.fasta'.format(protein),'w') as f:
        for h, seq in selected:
            f.write('>{}\n'.format(h))
            f.write('{}\n'.format(seq))

In [23]:
# Write all proteins using multiple threads

def _helper(protein):
    return write_proteins(other_seqs, protein)


with ThreadPoolExecutor(max_workers=10) as executor:
    for i, result in enumerate(executor.map(_helper, [p for p in prot_names])):
        print(f"Processed {prot_names[i]}...")

Processed NSP1...
Processed NSP2...
Processed NSP3...
Processed NSP4...
Processed NSP5...
Processed NSP6...
Processed NSP7...
Processed NSP8...
Processed NSP9...
Processed NSP10...
Processed NSP11...
Processed NSP12...
Processed NSP13...
Processed NSP14...
Processed NSP15...
Processed NSP16...
Processed Spike...
Processed NS3...
Processed E...
Processed M...
Processed NS6...
Processed NS7a...
Processed NS7b...
Processed NS8...
Processed N...
